In [30]:
import json
import logging
import pandas as pd
from typing import List, Any
from haystack.dataclasses import ByteStream, Document
from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentCleaner, DocumentSplitter
import requests
from tqdm import tqdm
import re

In [31]:
# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

In [32]:
# Load the PDF URLs from the JSON file
with open('sources/pdf_urls.json', 'r') as file:
    pdf_urls = json.load(file)
pdf_urls = pdf_urls[:20] + pdf_urls[460:480] + pdf_urls[-20:]

In [81]:
#TODO: generate tags for each pdf file

In [82]:
#TODO: generate tags for each pdf file

In [83]:
#TODO: generate summary for each pdf file

In [84]:
#TODO: generate hyq for each pdf file

In [85]:
#TODO: generate hyq declarative for each pdf file

In [33]:
# Define the parser class
class OFASParser:
    def __init__(self):
        self.pdf_converter = PyPDFToDocument()
        self.cleaner = DocumentCleaner(
            remove_empty_lines=True,
            remove_extra_whitespaces=True,
            remove_repeated_substrings=False,
        )
        self.splitter = DocumentSplitter(
            split_by="sentence",
            split_length=5,
            split_overlap=1,
            split_threshold=4,
        )

    def clean_text(self, text: str) -> str:
        # Remove excess dots and formatting artifacts
        text = re.sub(r'\.{4,}', '', text)  # Remove sequences of three or more dots
        text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
        text = re.sub(r'\n+', '\n', text)  # Replace multiple newlines with a single newline
        text = text.strip()  # Remove leading and trailing whitespace
        return text

    def convert_to_documents(self, pdf_urls: List[dict]) -> List[dict]:
        documents = []
        for url in tqdm(pdf_urls, desc="Processing PDFs", unit="file"):
            # print(url)
            try:
                # Fetch the PDF content
                response = requests.get(url['url'])
                response.raise_for_status()

                # Check if the content is a PDF
                if response.headers.get('Content-Type') != 'application/pdf':
                    logger.warning(f"Skipping non-PDF content from {url['url']}")
                    continue

                pdf_content = response.content

                # Convert PDF content to Document objects
                byte_stream = ByteStream(data=pdf_content)
                result = self.pdf_converter.run(sources=[byte_stream])
                converted_docs = result["documents"]  # Ensure this is a list of Document objects

                # Clean documents
                cleaned_result = self.cleaner.run(documents=converted_docs)
                cleaned_docs = cleaned_result["documents"]  # Extract the list of cleaned Document objects

                # Split documents
                split_result = self.splitter.run(documents=cleaned_docs)
                split_docs = split_result["documents"]  # Extract the list of split Document objects

                # Create document format for each converted document
                for doc in cleaned_docs:
                    # Clean the text content
                    cleaned_text = self.clean_text(doc.content)

                    # Extract language from the URL
                    language = url['url'].split('/')[3]
                    document = {
                        "url": url['url'],
                        "text": cleaned_text,
                        "language": language,
                        "tags": url['tag'],  # Leave empty for now
                        "subtopics": url['subtopics'],  # Leave empty for now
                        "summary": "",  # Leave empty for now
                        "doctype": "context_doc",  # Constant value
                        "organizations": "OFAS",  # Constant value
                        "hyq": "",  # Leave empty for now
                        "hyq_declarative": ""  # Leave empty for now
                    }
                    documents.append(document)
            except requests.RequestException as e:
                logger.error(f"Failed to fetch PDF from {url}: {e}")
        return documents

In [34]:
# Initialize the parser
ofas_parser = OFASParser()

In [35]:
# Process the PDF URLs
def process_pdfs():
    # Convert PDFs to documents
    documents = ofas_parser.convert_to_documents(pdf_urls)
    
    # Output the documents
    for doc in documents[:3]:
        print(doc)
    
    return documents


In [36]:
# Run the processing function
documents = process_pdfs()

Processing PDFs: 100%|██████████| 60/60 [02:47<00:00,  2.80s/file]

{'url': 'https://sozialversicherungen.admin.ch/it/d/6905/download', 'text': '1 Panoramica delle direttive sui contributi, stato 1º gennaio 2021 Il presente documento offre una panoramica delle direttive sui contributi AVS/AI/IPG/AD e fornisce agli utenti uno strumento per trovare rapidamente le disposizioni delle direttive determinanti per una precisa questione. Per motivi di chiarezza, singoli temi non ordinari sono qui tralasciati o trattati solo a grandi linee. Le direttive principali a cui si fa riferiment o nella presente panoramica sono le Direttive sull’obbligo assicurativo nell’AVS/AI (DOA), le Direttive sul salario determinante nell’AVS/AI e nelle IPG (DSD), le Direttive sui contributi dei lavoratori indipendenti e delle persone senza attività lucrativa nell’AVS/AI e nelle IPG (DIN) e le Direttive sulla riscossione dei contributi nell’AVS/AI e nelle IPG (DRC). Le DOA determinano quali persone sono soggette all’AVS/AI/IPG/AD obbligatoria. Le DSD elencano i redditi dei salariati

In [37]:
# Save documents to a CSV file
df = pd.DataFrame(documents)  # Create a DataFrame from the list of documents
df.to_csv('output/ofas.csv', index=False)  # Save to CSV without the index

In [1]:
pip install langfuse

Note: you may need to restart the kernel to use updated packages.


In [4]:
from langfuse import Langfuse

langfuse = Langfuse(
  secret_key="sk-lf-bbd40247-0163-4759-9ffe-dc786b8986dd",
  public_key="pk-lf-f44bb910-bf78-4d00-81e4-21ed63fdb166",
  host="http://localhost:3000"
)

In [5]:
async def test_langfuse_connection(langfuse_client):
    try:
        # Attempt to make a simple request to the Langfuse API
        response = await langfuse_client.ping()  # Assuming there's a ping endpoint
        if response.status_code == 200:
            logger.info("Langfuse connection successful.")
            return True
        else:
            logger.error(f"Langfuse connection failed with status code: {response.status_code}")
            return False
    except Exception as e:
        logger.error(f"Error connecting to Langfuse: {e}")
        return False

In [8]:
connection_successful = await test_langfuse_connection(langfuse)
if connection_successful:
    logger.info("Langfuse is ready to use.")
else:
    logger.error("Langfuse connection could not be established.")

NameError: name 'logger' is not defined